# 📱 Google Colab APK & AAB Builder untuk Aplikasi Android
Notebook ini dikhususkan untuk membangun / kompilasi **APK** (Android Package) dan **AAB** (Android App Bundle) dari ZIP Source Code aplikasi secara otomatis menggunakan **Java 17, Gradle 9.3.1, dan Android SDK 34** di Google Colab.

### Langkah 1: Hubungkan ke Google Drive
Jalankan sel ini untuk menautkan akun Google Drive Anda.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')
print('✅ Google Drive berhasil dihubungkan!')

### Langkah 2: Ekstraksi Source Code dari Google Drive
Pastikan Anda telah mengunggah file ZIP source code aplikasi ke Google Drive Anda (misalnya dengan nama `source_code.zip`).

In [ ]:
# Ubah nama file ZIP jika nama file di Drive Anda berbeda
ZIP_FILE_NAME = 'source_code.zip'
ZIP_PATH = f'/content/drive/MyDrive/{ZIP_FILE_NAME}'
EXTRACT_DIR = '/content/app_source'

import zipfile
import os
import shutil

if os.path.exists(ZIP_PATH):
    print(f'📦 Mengekstrak {ZIP_PATH} ke {EXTRACT_DIR}...')
    if os.path.exists(EXTRACT_DIR):
        shutil.rmtree(EXTRACT_DIR)
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    
    # Deteksi lokasi folder utama proyek (tempat settings.gradle.kts atau build.gradle.kts berada)
    project_root = EXTRACT_DIR
    for root, dirs, files in os.walk(EXTRACT_DIR):
        if 'settings.gradle.kts' in files or 'build.gradle.kts' in files:
            project_root = root
            break
    
    print(f'✅ Ekstraksi selesai! Root proyek ditemukan di: {project_root}')
    with open('/content/project_root.txt', 'w') as f:
        f.write(project_root)
else:
    print(f'❌ File {ZIP_PATH} tidak ditemukan di Google Drive.')
    print('Silakan pastikan file ZIP sudah diupload ke Google Drive (Drive Saya) dengan nama source_code.zip')

### Langkah 3: Persiapan Environment Build (Java 17, Gradle 9.3.1 & Android SDK 34)
Sel ini menginstall Java 17, mengunduh Gradle 9.3.1 resmi (yang dibutuhkan oleh Android Gradle Plugin 9.x), serta memasang Android SDK 34.

In [ ]:
import os

print('⏳ Memasang OpenJDK 17...')
!apt-get update -qq > /dev/null
!apt-get install -y openjdk-17-jdk wget unzip -qq > /dev/null
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'

print('⏳ Mengunduh dan memasang Gradle 9.3.1 resmi...')
!wget -q https://services.gradle.org/distributions/gradle-9.3.1-bin.zip -O /content/gradle-9.3.1-bin.zip
!mkdir -p /opt/gradle
!unzip -q -o /content/gradle-9.3.1-bin.zip -d /opt/gradle
os.environ['PATH'] = f"/opt/gradle/gradle-9.3.1/bin:{os.environ['PATH']}"

print('⏳ Mengkonfigurasi Android SDK 34 & Command-line Tools...')
!mkdir -p /root/Android/Sdk/cmdline-tools
!wget -q https://dl.google.com/android/repository/commandlinetools-linux-9477386_latest.zip -O /content/cmdline-tools.zip
!unzip -q -o /content/cmdline-tools.zip -d /root/Android/Sdk/cmdline-tools
!mv /root/Android/Sdk/cmdline-tools/cmdline-tools /root/Android/Sdk/cmdline-tools/latest 2>/dev/null || true

# Menerima Lisensi Android SDK & Install Platforms
!yes | /root/Android/Sdk/cmdline-tools/latest/bin/sdkmanager --licenses > /dev/null 2>&1
!/root/Android/Sdk/cmdline-tools/latest/bin/sdkmanager "platforms;android-34" "build-tools;34.0.0" "platform-tools" > /dev/null 2>&1

os.environ['ANDROID_HOME'] = '/root/Android/Sdk'
print('🎉 Environment berhasil disiapkan: Java 17, Gradle 9.3.1, & Android SDK 34 Siap!')

### Langkah 4: Proses Build APK & AAB (assembleDebug & bundleDebug)
Menjalankan proses kompilasi proyek menggunakan Gradle 9.3.1 untuk menghasilkan **file APK** dan **file AAB** sekaligus.

In [ ]:
import os

if os.path.exists('/content/project_root.txt'):
    with open('/content/project_root.txt', 'r') as f:
        project_root = f.read().strip()
    
    os.chdir(project_root)
    print(f'📍 Berada di lokasi proyek: {os.getcwd()}')
    
    # Buat file local.properties untuk Gradle
    with open('local.properties', 'w') as f:
        f.write('sdk.dir=/root/Android/Sdk\n')
    
    # Tambahkan jvmargs di gradle.properties agar alokasi memori optimal
    with open('gradle.properties', 'a') as f:
        f.write('\norg.gradle.jvmargs=-Xmx3072m -XX:MaxMetaspaceSize=768m\n')
    
    print('🚀 Memulai proses kompilasi APK & AAB (Gradle 9.3.1 assembleDebug bundleDebug)...')
    !/opt/gradle/gradle-9.3.1/bin/gradle assembleDebug bundleDebug --no-daemon
else:
    print('❌ Root proyek tidak ditemukan. Silakan jalankan Langkah 2 terlebih dahulu.')

### Langkah 5: Simpan Hasil APK & AAB ke Google Drive
Menyalinkan file APK (`AplikasiZakatHutang-debug.apk`) dan file AAB (`AplikasiZakatHutang-debug.aab`) yang telah dibuat langsung ke Google Drive Anda.

In [ ]:
import shutil
import os

if os.path.exists('/content/project_root.txt'):
    with open('/content/project_root.txt', 'r') as f:
        project_root = f.read().strip()
    
    # 1. Cari File APK
    found_apk = None
    possible_apk_paths = [
        os.path.join(project_root, 'app/build/outputs/apk/debug/app-debug.apk'),
        os.path.join(project_root, 'build/outputs/apk/debug/app-debug.apk')
    ]
    for p in possible_apk_paths:
        if os.path.exists(p):
            found_apk = p
            break
    if not found_apk:
        for root, dirs, files in os.walk(project_root):
            for file in files:
                if file.endswith('.apk'):
                    found_apk = os.path.join(root, file)
                    break
            if found_apk: break

    # 2. Cari File AAB
    found_aab = None
    possible_aab_paths = [
        os.path.join(project_root, 'app/build/outputs/bundle/debug/app-debug.aab'),
        os.path.join(project_root, 'build/outputs/bundle/debug/app-debug.aab')
    ]
    for p in possible_aab_paths:
        if os.path.exists(p):
            found_aab = p
            break
    if not found_aab:
        for root, dirs, files in os.walk(project_root):
            for file in files:
                if file.endswith('.aab'):
                    found_aab = os.path.join(root, file)
                    break
            if found_aab: break

    # Copy ke Drive
    drive_apk_target = '/content/drive/MyDrive/AplikasiZakatHutang-debug.apk'
    drive_aab_target = '/content/drive/MyDrive/AplikasiZakatHutang-debug.aab'
    
    print('--------------------------------------------------')
    if found_apk and os.path.exists(found_apk):
        shutil.copy(found_apk, drive_apk_target)
        print(f'🎉 SUKSES! File APK disimpan di: {drive_apk_target}')
    else:
        print('❌ File APK tidak ditemukan.')

    if found_aab and os.path.exists(found_aab):
        shutil.copy(found_aab, drive_aab_target)
        print(f'🎉 SUKSES! File AAB (Bundle) disimpan di: {drive_aab_target}')
    else:
        print('❌ File AAB tidak ditemukan.')
    print('--------------------------------------------------')
else:
    print('❌ Root proyek tidak ditemukan.')